# AdaBoost Regressor - Insurance Charges Prediction

## 1. Load Insurance Dataset
Load the insurance dataset from the AdaBoost regression folder and verify the file path.

In [11]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

data_path = Path("insurance.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path.resolve()}")

df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.head()

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Inspect and Clean Data
Handle missing values, fix data types, and normalize categorical columns.

In [12]:
df.info()

numeric_cols = ["age", "bmi", "children", "charges"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["sex", "smoker", "region"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

df = df.dropna().reset_index(drop=True)
print("After cleaning:", df.shape)
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
After cleaning: (1338, 7)
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


## 3. Feature Engineering and Encoding
Prepare features, encode categorical variables, and set the target column.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["charges"])
y = df["charges"].copy()

categorical_cols = ["sex", "smoker", "region"]
numeric_features = ["age", "bmi", "children"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

## 4. Train/Validation/Test Split
Split the dataset into train/validation/test sets with a fixed random seed.

In [14]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (936, 6) Val: (201, 6) Test: (201, 6)


## 5. Train AdaBoost Regressor
Fit an AdaBoost regressor and evaluate performance on the test set.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "regressor",
            AdaBoostRegressor(
                estimator=DecisionTreeRegressor(max_depth=3, random_state=42),
                n_estimators=300,
                learning_rate=0.05,
                random_state=42,
            ),
        ),
    ]
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
metrics = {
    "mae": mean_absolute_error(y_test, preds),
    "r2": r2_score(y_test, preds),
}
metrics

{'mae': 4380.730408984584, 'r2': 0.8173949547770396}

## 6. Save the Model
Persist the trained model to disk for use in the Streamlit app.

In [16]:
import joblib

joblib.dump(model, "adaboost_insurance_model.pkl")
print("Saved model to adaboost_insurance_model.pkl")

Saved model to adaboost_insurance_model.pkl
